# Indian agriculture 1966-2017: national trends

**Notebook 2.** Here I look at how area, production and yield changed over the 52 years for the main crops, by adding up all the districts in each year.

I'm using the cleaned file from notebook 1 (`crop_data_clean.csv`), so that notebook needs to be run first.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## Loading the cleaned data

In [ ]:
crop_data = pd.read_csv("../data/clean/crop_data_clean.csv")
crop_data.shape

In [ ]:
crop_data.head()

In [ ]:
# years covered and number of crops
print(crop_data["year"].min(), "to", crop_data["year"].max())
print(crop_data["crop"].nunique(), "crops")

## Choosing the crops

Twenty crops is too many to read on charts, so I'm taking eight important ones: rice, wheat and maize (cereals), sorghum and pearl millet (millets), chickpea (pulses), groundnut (oilseed) and cotton (cash crop).

I'm leaving sugarcane out because its units look different from the other crops, and minor pulses because it has a lot of missing data.

In [ ]:
main_crops = ["Rice", "Wheat", "Maize", "Sorghum", "Pearl millet", "Chickpea", "Groundnut", "Cotton"]

main_data = crop_data[crop_data["crop"].isin(main_crops)]
main_data.shape

## Yearly totals

For each crop and year I add up the area and production of all districts. Yield is then total production divided by total area (times 1000 for kg per hectare).

I'm not averaging the district yields, because the strange yields I found in notebook 1 would distort the average. In totals, those tiny districts hardly matter.

I only use rows where both area and production are known, so the two totals cover the same districts.

In [ ]:
# only rows where both area and production are known
main_data = main_data.dropna(subset=["area", "production"])

yearly_totals = main_data.groupby(["crop", "year"])[["area", "production"]].sum().reset_index()

# 1000 tons / 1000 ha * 1000 = kg per ha
yearly_totals["crop_yield"] = yearly_totals["production"] / yearly_totals["area"] * 1000

yearly_totals.head()

In [ ]:
# every crop should have 52 years
yearly_totals.groupby("crop").size()

## Area, production and yield over time

A small function so I don't repeat the same chart three times. Each crop gets its own panel with its own scale, because rice and chickpea are on very different levels.

In [ ]:
def plot_by_crop(column, title, ylabel):
    chart = sns.relplot(data=yearly_totals, x="year", y=column, col="crop", col_wrap=4,
                        kind="line", height=2.6, aspect=1.2, facet_kws={"sharey": False})
    chart.set_titles("{col_name}")
    chart.set_axis_labels("year", ylabel)
    chart.figure.suptitle(title, y=1.03)
    plt.show()

In [ ]:
plot_by_crop("area", "Area grown by crop", "area (1000 ha)")

In [ ]:
plot_by_crop("production", "Production by crop", "production (1000 tons)")

In [ ]:
plot_by_crop("crop_yield", "Yield by crop", "yield (kg per ha)")

Yield goes up for every crop, but area doesn't. Wheat and cotton area clearly grew, while sorghum area dropped a lot. I'll put numbers on this further down.

## Rice and wheat: the Green Revolution

The Green Revolution (new high-yielding seeds, more fertiliser and irrigation) started in the late 1960s, mostly for wheat and rice. The data starts in 1966, so I can see its early years.

Yearly yields jump around with the weather, so I added a 5-year rolling average to see the trend more clearly.

In [ ]:
rice_wheat = yearly_totals[yearly_totals["crop"].isin(["Rice", "Wheat"])].copy()

# 5 year average, centered on each year
rice_wheat["smooth_yield"] = rice_wheat.groupby("crop")["crop_yield"].transform(
    lambda values: values.rolling(5, center=True).mean())

plt.figure(figsize=(10, 5))
sns.lineplot(data=rice_wheat, x="year", y="crop_yield", hue="crop", alpha=0.4, legend=False)
sns.lineplot(data=rice_wheat, x="year", y="smooth_yield", hue="crop", linewidth=2.5)

# shading the early years
plt.axvspan(1966, 1980, color="grey", alpha=0.15)
plt.text(1967, 3300, "early Green\nRevolution years", fontsize=9)

plt.title("Rice and wheat yield (thin line = each year, thick line = 5 year average)")
plt.xlabel("year")
plt.ylabel("yield (kg per ha)")
plt.show()

In [ ]:
# wheat and rice yield in a few years
rice_wheat[rice_wheat["year"].isin([1966, 1970, 1980, 1990, 2000, 2010, 2017])][["crop", "year", "crop_yield"]].round(0)

Wheat yield went from about 880 kg/ha in 1966 to about 1,300 in 1970, and rice from about 860 to about 1,100. 1966 might have been a bad year, so part of that jump could just be recovery, but the climb continues for decades after, so it's more than that.

## How fast did things grow?

To compare crops I want a growth rate in % per year. I fit a straight line to the log of the values with NumPy (`np.polyfit`). The slope of that line gives the average yearly growth.

This is better than comparing only the first and last year, because one bad year at either end could change the answer. The line uses all 52 years.

In [ ]:
growth_rows = []

for crop in main_crops:
    one_crop = yearly_totals[yearly_totals["crop"] == crop]
    row = {"crop": crop}

    for column in ["area", "production", "crop_yield"]:
        slope = np.polyfit(one_crop["year"], np.log(one_crop[column]), 1)[0]
        row[column] = (np.exp(slope) - 1) * 100

    growth_rows.append(row)

growth_table = pd.DataFrame(growth_rows).set_index("crop").round(2)
growth_table.columns = ["area growth %", "production growth %", "yield growth %"]

growth_table.sort_values("yield growth %", ascending=False)

Production growth is roughly area growth plus yield growth, which is a good sign that the numbers are consistent (for rice: 0.39 + 2.04 is about 2.44).

In [ ]:
growth_table[["area growth %", "yield growth %"]].plot(kind="bar", figsize=(10, 5))

plt.axhline(0, color="black", linewidth=0.8)
plt.title("Yield grew for every crop, but area shrank for sorghum, pearl millet and groundnut")
plt.xlabel("")
plt.ylabel("growth per year (%)")
plt.xticks(rotation=0)
plt.show()

## Start vs end

Another simple way to see the change: average yield in the first three years (1966-68) against the last three (2015-17). I use three years at each end so one bad year doesn't decide the result.

In [ ]:
start_years = yearly_totals[yearly_totals["year"].between(1966, 1968)]
end_years = yearly_totals[yearly_totals["year"].between(2015, 2017)]

comparison = pd.DataFrame({
    "yield 1966-68": start_years.groupby("crop")["crop_yield"].mean(),
    "yield 2015-17": end_years.groupby("crop")["crop_yield"].mean(),
})
comparison["times higher"] = comparison["yield 2015-17"] / comparison["yield 1966-68"]

comparison.sort_values("times higher", ascending=False).round(2)

In [ ]:
# same idea for area: how much did the area change?
area_comparison = pd.DataFrame({
    "area 1966-68": start_years.groupby("crop")["area"].mean(),
    "area 2015-17": end_years.groupby("crop")["area"].mean(),
})
area_comparison["change %"] = (area_comparison["area 2015-17"] / area_comparison["area 1966-68"] - 1) * 100

area_comparison.sort_values("change %").round(0)

## Yield by decade

In [ ]:
yearly_totals["decade"] = (yearly_totals["year"] // 10 * 10).astype(str) + "s"

decade_yield = yearly_totals.pivot_table(index="crop", columns="decade", values="crop_yield")

plt.figure(figsize=(9, 5))
sns.heatmap(decade_yield, annot=True, fmt=".0f", cmap="YlGn")
plt.title("Average yield (kg per ha) by decade")
plt.xlabel("")
plt.ylabel("")
plt.show()

Note: the 1960s only has 4 years in the data (1966-69) and the 2010s has 8 (2010-17), so those two columns cover fewer years than the others.

## What I found

- **Yields went up for all 8 crops.** Wheat yield in 2015-17 is about 3.3 times what it was in 1966-68, and rice is about 2.7 times higher. Cotton (3.5 times) and pearl millet (3.4 times) rose even more. Sorghum (1.5 times) and chickpea (1.7 times) rose the least.
- **Five crops grew about 2 to 2.7% a year in yield:** cotton, pearl millet, maize, wheat and rice. Groundnut (1.5%), sorghum (1.1%) and chickpea (1.0%) grew more slowly.
- **Rice and wheat grew in different ways.** Rice area grew only 0.4% a year, so almost all of its production growth came from higher yield. Wheat got both: area grew 1.3% a year (it more than doubled over the 52 years) and yield grew 2.3% a year.
- **Some crops lost land.** Sorghum area fell by about two thirds, pearl millet by about 40% and groundnut by about a third. Sorghum production is now lower than in 1966-68, even though its yield went up, because so much land left the crop.
- **The Green Revolution shows up in the data.** Wheat and rice yields climb steeply from the late 1960s. Part of the 1966 to 1970 jump may be recovery from a bad year in 1966, but yields keep rising for decades after that, so it is more than recovery.

**Limits:** this is only the 20 states in the ICRISAT data, not all of India, and it ends in 2017. The yearly dips in the charts come from weather, which I look at in notebook 4. I only describe what happened here, not why.

**Next notebook:** regional gaps. Did the districts with low yields catch up with the best ones?